<a href="https://colab.research.google.com/github/Akanshajoshiiii/NLP_LAB/blob/main/joint_classi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

joint token classification & text classification e,g intent classifiaction and slot filling

In [ ]:
from datasets import load_dataset
dataset=load_dataset("tuetschek/atis")

In [ ]:
dataset['train'][2]

{'id': 2,
 'intent': 'flight_time',
 'text': 'what is the arrival time in san francisco for the 755 am flight leaving washington',
 'slots': 'O O O B-flight_time I-flight_time O B-fromloc.city_name I-fromloc.city_name O O B-depart_time.time I-depart_time.time O O B-fromloc.city_name'}

In [ ]:
#problems in data
#everything is words , needs to replaced in numbers
# this data is also not tokenised
#

In [ ]:
def preprocess(example):
    return{'tokens':example['text'].split(),
           'slot_labels':example['slots'].split()
    }

In [ ]:
dataset=dataset.map(preprocess)

Map: 100%|██████████| 893/893 [00:00<00:00, 18871.84 examples/s]


In [ ]:
from collections import Counter

counter = Counter()

for example in dataset['train']:
    for token in example['tokens']:
        counter.update([token.lower()])

vocab = {'PAD': 0, 'UNK': 1}

for word in counter:
    vocab[word] = len(vocab)

In [ ]:
counter1=Counter()
for example in dataset['train']:
    counter1.update(slot_label.lower() for slot_label in example['slot_labels'])
slot2index={}
for slot_label in counter1:
    slot2index[slot_label]=len(slot2index)

In [ ]:
iintentlabels=set(example['intent'] for example in dataset['train'])
intent2index={intent:index for index, intent in enumerate(iintentlabels)}

In [ ]:
intent2index

{'flight+airfare': 0,
 'aircraft+flight+flight_no': 1,
 'capacity': 2,
 'airport': 3,
 'distance': 4,
 'city': 5,
 'cheapest': 6,
 'ground_service+ground_fare': 7,
 'airline+flight_no': 8,
 'ground_fare': 9,
 'flight_time': 10,
 'airline': 11,
 'flight_no': 12,
 'ground_service': 13,
 'airfare': 14,
 'abbreviation': 15,
 'quantity': 16,
 'airfare+flight_time': 17,
 'aircraft': 18,
 'flight': 19,
 'meal': 20,
 'restriction': 21}

In [ ]:
def encode(example):
    return {
        'input_ids': [vocab.get(token.lower(), 1)
                      for token in example['tokens']],

        'slot_ids': [slot2index.get(slot_label.lower(), 0)
                     for slot_label in example['slot_labels']],

        'intent_id': intent2index.get(example['intent'], 22)
    }

In [ ]:
dataset = dataset.map(encode)

Map: 100%|██████████| 893/893 [00:00<00:00, 9855.42 examples/s]


In [ ]:
dataset['train'][0]

{'id': 0,
 'intent': 'flight',
 'text': 'i want to fly from boston at 838 am and arrive in denver at 1110 in the morning',
 'slots': 'O O O O O B-fromloc.city_name O B-depart_time.time I-depart_time.time O O O B-toloc.city_name O B-arrive_time.time O O B-arrive_time.period_of_day',
 'tokens': ['i',
  'want',
  'to',
  'fly',
  'from',
  'boston',
  'at',
  '838',
  'am',
  'and',
  'arrive',
  'in',
  'denver',
  'at',
  '1110',
  'in',
  'the',
  'morning'],
 'slot_labels': ['O',
  'O',
  'O',
  'O',
  'O',
  'B-fromloc.city_name',
  'O',
  'B-depart_time.time',
  'I-depart_time.time',
  'O',
  'O',
  'O',
  'B-toloc.city_name',
  'O',
  'B-arrive_time.time',
  'O',
  'O',
  'B-arrive_time.period_of_day'],
 'input_ids': [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 8, 15, 13, 16, 17],
 'slot_ids': [0, 0, 0, 0, 0, 1, 0, 2, 3, 0, 0, 0, 4, 0, 5, 0, 0, 6],
 'intent_id': 19}

In [ ]:
import torch

def padding(batch):
    maxlen = max(len(example['input_ids']) for example in batch)

    input_ids = []
    slot_ids = []
    intent_ids = []
    mask = []

    for example in batch:
        L = len(example['input_ids'])
        padlen = maxlen - L

        input_ids.append(example['input_ids'] + [0] * padlen)
        slot_ids.append(example['slot_ids'] + [0] * padlen)
        intent_ids.append(example['intent_id'])   # fixed key

        mask.append([1] * L + [0] * padlen)       # fixed syntax

    return (
        torch.tensor(input_ids),
        torch.tensor(slot_ids),
        torch.tensor(intent_ids),
        torch.tensor(mask, dtype=torch.float)
    )

In [ ]:
from torch.utils.data import DataLoader

train_loader=DataLoader(dataset['train'],batch_size=32,shuffle=True,collate_fn=padding)
test_loader=DataLoader(dataset['train'],batch_size=32,shuffle=True,collate_fn=padding)

In [ ]:
import torch
import torch.nn as nn

class LSTMCellScratch(nn.Module):
    def __init__(self, embedding_dim, hidden_dim):
        super().__init__()

        self.hidden_dim = hidden_dim

        # Gates
        self.input_layer = nn.Linear(embedding_dim + hidden_dim, hidden_dim)
        self.output_layer = nn.Linear(embedding_dim + hidden_dim, hidden_dim)
        self.forget_layer = nn.Linear(embedding_dim + hidden_dim, hidden_dim)
        self.candidate_layer = nn.Linear(embedding_dim + hidden_dim, hidden_dim)

    def forward(self,input,h,c):
        concatenated = torch.cat((input,h), dim=1)

        i = torch.sigmoid(self.input_layer(concatenated))      # input gate
        f = torch.sigmoid(self.forget_layer(concatenated))     # forget gate
        o = torch.sigmoid(self.output_layer(concatenated))     # output gate
        cand = torch.tanh(self.candidate_layer(concatenated))     # candidate

        c = i*cand+f*c
        h = o * torch.tanh(c)

        return h, c